# 🌾 Pasture Biomass Prediction Pipeline (Foundation & End-to-End Models)
This notebook implements an advanced, end-to-end machine learning pipeline to predict five pasture biomass targets from RGB images using the CSIRO Image2Biomass dataset.

### Key Features:
1. **GPU Acceleration**: Fully utilizes your **NVIDIA GeForce RTX 4050 GPU** for deep learning inference and training.
2. **DINOv2 Backbone**: Extracts rich 384-dimensional features using **Meta's DINOv2** (`facebook/dinov2-small`) self-supervised vision transformer via Hugging Face.
3. **SegFormer Pasture Segmentation**: Uses a pretrained **SegFormer** (`nvidia/segformer-b0-finetuned-ade-512-512`) for zero-shot pasture segmentation.
4. **Depth Anything**: Uses **Depth Anything** (`LiheYoung/depth-anything-small-hf`) for high-resolution monocular depth estimation.
5. **Fine-Tuned ResNet50 (End-to-End)**: Trains a deep convolutional neural network directly on the pasture images to predict biomass.
6. **Robust Caching**: Automatically caches extracted features to `features_foundation.csv` to ensure instant subsequent runs.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit
import xgboost as xgb
import joblib

# Hugging Face Transformers & Torchvision
from transformers import (
    SegformerImageProcessor, 
    SegformerForSemanticSegmentation,
    AutoImageProcessor,
    AutoModelForDepthEstimation,
    AutoModel
)
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

# Define base directories
base_dir = r"c:\Users\ratha\OneDrive\Desktop\datavidwan\New folder (2)\csiro-biomass"
print("Base Directory:", base_dir)

## Stage 1: Data Audit & Split
We load the metadata, pivot it to wide format, and perform a group-aware train/validation split by `Sampling_Date` to prevent data leakage.

In [ ]:
from src.dataset import load_and_pivot_metadata, get_group_split, TARGET_NAMES

# Load pivoted metadata
df_pivoted = load_and_pivot_metadata(base_dir)
print(f"Total Unique Images: {len(df_pivoted)}")

# Split data
train_df, val_df = get_group_split(df_pivoted, test_size=0.2, random_state=42)
print(f"Train set size: {len(train_df)} | Validation set size: {len(val_df)}")
print("Date overlap between splits (should be 0):", len(set(train_df['Sampling_Date']).intersection(set(val_df['Sampling_Date']))))

## Stage 2: Zero-Shot Pasture Segmentation (SegFormer)
We use SegFormer trained on ADE20K. We map classes:
- **Vegetation**: `plant` (12), `grass` (29), `flower` (140)
- **Soil/Earth**: `dirt` (94), `earth` (13), `sand` (11)
We compute the vegetation coverage, green ratio, and dead ratio.

In [ ]:
print("Loading SegFormer model...")
seg_processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
seg_model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512").to(device)
seg_model.eval()

def segment_pasture(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    inputs = seg_processor(images=image_rgb, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = seg_model(**inputs)
        logits = outputs.logits
        # Upsample logits to original image size
        upsampled_logits = nn.functional.interpolate(
            logits,
            size=image_rgb.shape[:2],
            mode="bilinear",
            align_corners=False
        )
        pred_mask = upsampled_logits.argmax(dim=1)[0].cpu().numpy()
        
    # Map classes (updated for pasture-specific ADE20K predictions)
    # Vegetation: tree (4), grass (9), plant (17), field (29), mountain (16)
    veg_mask = np.isin(pred_mask, [4, 9, 17, 29, 16])
    # Soil/Dead: everything else (e.g. wall, earth, path, etc.)
    soil_mask = ~veg_mask
    
    total_pixels = pred_mask.size
    veg_coverage = np.sum(veg_mask) / total_pixels
    soil_coverage = np.sum(soil_mask) / total_pixels
    
    # Ratios
    green_ratio = veg_coverage / (veg_coverage + soil_coverage + 1e-6)
    dead_ratio = soil_coverage / (veg_coverage + soil_coverage + 1e-6)
    
    return pred_mask, veg_coverage, green_ratio, dead_ratio

# Test on a sample image
sample_row = df_pivoted.iloc[0]
sample_path = os.path.join(base_dir, sample_row['image_path'])
pred_mask, veg_cov, green_r, dead_r = segment_pasture(sample_path)

print(f"Sample: {sample_row['image_path']}")
print(f"Veg Coverage: {veg_cov:.4f} | Green Ratio: {green_r:.4f} | Dead Ratio: {dead_r:.4f}")

# Visualize
img = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(pred_mask, cmap="tab20")
axes[1].set_title("SegFormer Classes")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## Stage 3: Monocular Depth Estimation (Depth Anything)
We use Depth Anything (`LiheYoung/depth-anything-small-hf`) to estimate depth maps of the canopy and extract mean, standard deviation, and variance of the depth.

In [ ]:
print("Loading Depth Anything model...")
depth_processor = AutoImageProcessor.from_pretrained("LiheYoung/depth-anything-small-hf")
depth_model = AutoModelForDepthEstimation.from_pretrained("LiheYoung/depth-anything-small-hf").to(device)
depth_model.eval()

def estimate_depth(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    inputs = depth_processor(images=image_rgb, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = depth_model(**inputs)
        depth = outputs.predicted_depth
        # Interpolate to original size
        depth = nn.functional.interpolate(
            depth.unsqueeze(1),
            size=image_rgb.shape[:2],
            mode="bilinear",
            align_corners=False
        ).squeeze(1)[0].cpu().numpy()
        
    # Calculate stats
    d_mean = np.mean(depth)
    d_std = np.std(depth)
    d_var = np.var(depth)
    
    return depth, d_mean, d_std, d_var

# Test depth map
depth_map, d_mean, d_std, d_var = estimate_depth(sample_path)
print(f"Depth Mean: {d_mean:.4f} | Depth Std: {d_std:.4f} | Depth Var: {d_var:.4f}")

plt.figure(figsize=(8, 5))
plt.imshow(depth_map, cmap="plasma")
plt.colorbar(label="Relative Depth")
plt.title("Depth Anything Map")
plt.axis("off")
plt.show()

## Stage 4: Visual Representation (DINOv2)
We use Meta's DINOv2 (`facebook/dinov2-small`) via Hugging Face `transformers` to extract a rich 384-dimensional feature vector representing the overall structure of each pasture image.

In [ ]:
print("Loading DINOv2 model...")
dino_processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dinov2_model = AutoModel.from_pretrained("facebook/dinov2-small", output_attentions=True).to(device)
dinov2_model.eval()

def extract_dinov2_features(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    inputs = dino_processor(images=image_rgb, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = dinov2_model(**inputs)
    # Extract the CLS token (index 0) from the sequence output
    return outputs.last_hidden_state[0, 0].cpu().numpy()

# Test feature extraction
dino_feats = extract_dinov2_features(sample_path)
print(f"DINOv2 Features shape: {dino_feats.shape}")

## Stage 5: Combined Feature Extraction with Caching
We extract features from all images. To save time on subsequent runs, we cache the results into `features_foundation.csv`.

In [ ]:
cache_path = r"c:\Users\ratha\OneDrive\Desktop\datavidwan\New folder (2)\features_foundation.csv"

if os.path.exists(cache_path):
    print("Loading features from cache...")
    features_df = pd.read_csv(cache_path)
    print("Features loaded! Shape:", features_df.shape)
else:
    print("Extracting features from scratch (this may take a few minutes)...")
    records = []
    for idx, row in tqdm(df_pivoted.iterrows(), total=len(df_pivoted)):
        img_path = os.path.join(base_dir, row['image_path'])
        
        try:
            # 1. SegFormer
            _, veg_cov, green_r, dead_r = segment_pasture(img_path)
            
            # 2. Depth Anything
            _, d_mean, d_std, d_var = estimate_depth(img_path)
            
            # 3. DINOv2
            dino_feats = extract_dinov2_features(img_path)
            
            # Create feature record
            record = {
                'image_path': row['image_path'],
                'Sampling_Date': row['Sampling_Date'],
                'veg_coverage': veg_cov,
                'green_ratio': green_r,
                'dead_ratio': dead_r,
                'depth_mean': d_mean,
                'depth_std': d_std,
                'depth_var': d_var,
                'Height_Ave_cm': row['Height_Ave_cm'],
                'Pre_GSHH_NDVI': row['Pre_GSHH_NDVI']
            }
            # Add DINOv2 features
            for f_idx, val in enumerate(dino_feats):
                record[f'dino_{f_idx}'] = val
                
            # Add target variables
            for target in TARGET_NAMES:
                record[target] = row[target]
                
            records.append(record)
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            
    features_df = pd.DataFrame(records)
    features_df.to_csv(cache_path, index=False)
    print("Features extracted and saved to cache! Shape:", features_df.shape)

## Stage 6: Model Training (Feature Extraction)
We train Random Forest, XGBoost, and a PyTorch MLP on the cached features (DINOv2 embeddings + SegFormer ratios + Depth stats + Metadata).

In [ ]:
# Define features list
dino_cols = [c for c in features_df.columns if c.startswith('dino_')]
feature_cols = [
    'veg_coverage', 'green_ratio', 'dead_ratio',
    'depth_mean', 'depth_std', 'depth_var',
    'Height_Ave_cm', 'Pre_GSHH_NDVI'
] + dino_cols

# Split data using GroupShuffleSplit on Sampling_Date
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(features_df, groups=features_df['Sampling_Date']))

train_data = features_df.iloc[train_idx].reset_index(drop=True)
val_data = features_df.iloc[val_idx].reset_index(drop=True)

X_train = train_data[feature_cols].values
y_train = train_data[TARGET_NAMES].values
X_val = val_data[feature_cols].values
y_val = val_data[TARGET_NAMES].values

print(f"Training features dimension: {X_train.shape[1]}")
print(f"Train size: {X_train.shape[0]} | Val size: {X_val.shape[0]}")

# PyTorch MLP Regressor definition
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, output_dim=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )
    def forward(self, x):
        return self.net(x)

def train_mlp(X_tr, y_tr, X_va, y_va):
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    
    model = MLPRegressor(X_tr.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    
    # DataLoaders
    train_ds = torch.utils.data.TensorDataset(torch.tensor(X_tr_s, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.float32))
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    
    for epoch in range(150):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_va_s, dtype=torch.float32).to(device)).cpu().numpy()
    return model, scaler, preds

# Helper to compute metrics
def evaluate_predictions(y_true, y_pred):
    metrics = {}
    for idx, target in enumerate(TARGET_NAMES):
        t_true = y_true[:, idx]
        t_pred = np.clip(y_pred[:, idx], 0, None)  # Biomass cannot be negative
        
        r2 = r2_score(t_true, t_pred)
        rmse = np.sqrt(mean_squared_error(t_true, t_pred))
        mae = mean_absolute_error(t_true, t_pred)
        metrics[target] = {'R2': r2, 'RMSE': rmse, 'MAE': mae}
    return metrics

all_results = {}

# 1. Random Forest
print("Training Random Forest...")
rf = RandomForestRegressor(n_estimators=150, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_val)
all_results["RF (DINOv2 Features)"] = evaluate_predictions(y_val, rf_preds)

# 2. XGBoost (multi-output via loop)
print("Training XGBoost...")
xgb_preds_list = []
for idx, target in enumerate(TARGET_NAMES):
    model_xgb = xgb.XGBRegressor(n_estimators=150, random_state=42, learning_rate=0.05)
    model_xgb.fit(X_train, y_train[:, idx])
    xgb_preds_list.append(model_xgb.predict(X_val))
xgb_preds = np.stack(xgb_preds_list, axis=1)
all_results["XGBoost (DINOv2 Features)"] = evaluate_predictions(y_val, xgb_preds)

# 3. PyTorch MLP
print("Training PyTorch MLP...")
mlp_model, mlp_scaler, mlp_preds = train_mlp(X_train, y_train, X_val, y_val)
all_results["MLP (DINOv2 Features)"] = evaluate_predictions(y_val, mlp_preds)

## Stage 6b: End-to-End Fine-Tuned ResNet50
We train a ResNet50 model end-to-end on the GPU. The weights of the network are fine-tuned directly on the pasture images to predict the 5 biomass targets.

In [ ]:
print("Setting up ResNet50 End-to-End Fine-Tuning...")

# 1. Define Dataset
class PastureImageDataset(Dataset):
    def __init__(self, df, base_dir, transform=None):
        self.df = df
        self.base_dir = base_dir
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.base_dir, row['image_path'])
        
        image = cv2.imread(img_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            image_rgb = self.transform(image_rgb)
            
        targets = row[TARGET_NAMES].values.astype(np.float32)
        return image_rgb, torch.tensor(targets)

# 2. Image Transformations (with data augmentation for training)
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Create DataLoaders
train_dataset = PastureImageDataset(train_df, base_dir, transform=train_transform)
val_dataset = PastureImageDataset(val_df, base_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# 4. Initialize ResNet50
resnet_model = resnet50(weights=ResNet50_Weights.DEFAULT)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 5) # 5 targets
resnet_model = resnet_model.to(device)

criterion = nn.MSELoss()
optimizer = optim.AdamW(resnet_model.parameters(), lr=1e-4, weight_decay=1e-4)

# 5. Training Loop
epochs = 20
best_val_loss = float('inf')
best_weights_path = r"c:\Users\ratha\OneDrive\Desktop\datavidwan\New folder (2)\best_resnet50_weights.pth"

print("Training ResNet50...")
for epoch in range(epochs):
    resnet_model.train()
    train_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = resnet_model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    # Validation
    resnet_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = resnet_model(images)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * images.size(0)
            
    train_loss /= len(train_dataset)
    val_loss /= len(val_dataset)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(resnet_model.state_dict(), best_weights_path)
        
    print(f"Epoch {epoch+1}/{epochs} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}")

# 6. Generate Predictions
resnet_model.load_state_dict(torch.load(best_weights_path))
resnet_model.eval()
all_resnet_preds = []
with torch.no_grad():
    for images, _ in val_loader:
        images = images.to(device)
        outputs = resnet_model(images)
        all_resnet_preds.append(outputs.cpu().numpy())
        
resnet_preds = np.concatenate(all_resnet_preds, axis=0)
all_results["ResNet50 (End-to-End)"] = evaluate_predictions(y_val, resnet_preds)
print("ResNet50 evaluation complete!")

### Performance Comparison
Let's display the metrics for each model and target.

In [ ]:
rows = []
for model_name, target_metrics in all_results.items():
    for target, metrics in target_metrics.items():
        rows.append({
            'Model': model_name,
            'Target': target,
            'R2': metrics['R2'],
            'RMSE (g)': metrics['RMSE'],
            'MAE (g)': metrics['MAE'],
            'RMSE (kg/ha)': metrics['RMSE'] * 47.619,
            'MAE (kg/ha)': metrics['MAE'] * 47.619
        })
df_comp = pd.DataFrame(rows)
display(df_comp)

# Calculate average RMSE across all targets
print("\n--- Average Validation RMSE across all 5 targets ---")
for model_name in all_results:
    avg_rmse = np.mean([all_results[model_name][t]['RMSE'] for t in TARGET_NAMES])
    print(f"{model_name} | Average Validation RMSE: {avg_rmse:.4f} g")

## Stage 7: Explainability (DINOv2 Self-Attention Maps)
We visualize the self-attention maps of the DINOv2 model's last layer to see what regions of the pasture the model is paying attention to.

In [ ]:
def get_dinov2_attention(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    w, h = image_rgb.shape[1], image_rgb.shape[0]
    
    inputs = dino_processor(images=image_rgb, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = dinov2_model(**inputs)
            
    # Get attentions from the last layer
    # Shape: [batch_size, num_heads, num_tokens, num_tokens]
    attentions = outputs.attentions[-1]
    nh = attentions.shape[1]
    
    # Keep only the attention of the CLS token to the other 256 tokens (16x16 grid)
    th_attn = attentions[0, :, 0, 1:].reshape(nh, 16, 16)
    
    # Average across heads
    mean_attn = torch.mean(th_attn, dim=0).cpu().numpy()
    
    # Resize to original image size
    mean_attn = cv2.resize(mean_attn, (w, h))
    
    return mean_attn

# Visualize attention map
attn_map = get_dinov2_attention(sample_path)
if attn_map is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(img)
    axes[0].set_title("Original Image")
    axes[0].axis("off")
    
    axes[1].imshow(img)
    axes[1].imshow(attn_map, cmap="jet", alpha=0.5)
    axes[1].set_title("DINOv2 Self-Attention Overlay")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Could not retrieve DINOv2 self-attention maps.")